In [1]:
import pandas as pd
import os
import warnings
import numpy as np
from src.config import *

In [2]:
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [ ]:
"""
2002년 1월 1일부터 2025년 12월 31일까지 연도별 구분된 가락시장 경락가격 정보를 하나의 csv 파일로 합쳐 최종적으로 price_배추_가락시장_2002_2025.csv로 구성

[입력]
- 파일 경로: RAW_DATA_DIR + 2026년 신규수집 데이터/농넷 가락시장 경락가격/배추/농넷 가락시장 경락가격_일단위_10kg_상특중하_배추_{연도}.xlsx

[경로 구성]
- 현재 경로와 파일명을 기반으로 file_paths 리스트에 현재 경로 + 파일명을 더한 전체 경로를 저장

[1차 가공]
- price 데이터를 하나의 데이터프레임으로 합치기 위해 concat_df 생성
- read_excel 함수를 통해 데이터프레임으로 변환 (read_excel 함수 사용시 engine 반드시 설정)
- 등급이 '상'인 데이터로 필터링
- 휴장일인 경우에는 평균가격이 '0'이고, 휴장일이라는 것을 명시적으로 나타내기 위해 NaN 결측치로 처리
- 등급과 휴장일에 대한 처리가 완료된 데이터프레임을 pd.concat([concat_df, filtered_df]) 처리

[2차 가공]
- DATE 컬럼을 데이터를 기반으로 시간순으로 정렬
- sort_values() 정렬 + 이후 pd.date_range()와의 타입 호환을 위해 datetime 변환이 필요

[3차 가공]
- DATE를 기준으로 가격 시계열 데이터를 만들고
- 중간에 빠진날짜를 생성하기 위해서 2002-01-01 ~ 2025-12-31까지 모든 날짜를 만든다
- 이때 데이터가 없는 날짜가 생기는데, 가격 데이터는 휴장일에 직전 거래일 가격을 그대로 유지하는 방식이기 때문에 선향 보간법이 아닌 ffill 방식을 사용

[출력]
- price_배추_가락시장_2002_2025.csv
"""
# 현재 경로 확인
data_folder = RAW_DATA_DIR + '2026년 신규수집 데이터/농넷 가락시장 경락가격/배추/'
print(data_folder)

# 파일명 확인
file_names = os.listdir(data_folder)
print(file_names[:1])

# 데이터 폴더 속 파일 각각의 경로 저장
file_paths = []
for file_name in file_names:
    file_paths.append(os.path.join(data_folder, file_name))

concat_df = pd.DataFrame()
for file_path in file_paths:
    # read_excel을 사용할땐, engine 파라미터를 명시해줘야한다.
    df = pd.read_excel(file_path, engine='openpyxl')
    
    # 등급명 == 상
    filtered_df = df[df['등급명'] == '상']
    
    # 평균가격 == 0 -> NaN처리 (휴장일) - 거래가 없기 때문에 NaN임
    filtered_df['평균가격'] = filtered_df['평균가격'].replace(0, np.nan)

    # concat
    concat_df = pd.concat([concat_df, filtered_df])

# DATE컬럼 datetime으로 변환
concat_df['DATE'] = pd.to_datetime(concat_df['DATE'])
concat_df = concat_df.sort_values('DATE').reset_index(drop=True)

price_series = concat_df.set_index('DATE')['평균가격']
full_idx = pd.date_range('2002-01-01', '2025-12-31')
price_filled = price_series.reindex(full_idx).ffill()

# csv export 전 NaN 값 확인
output_df = price_filled.reset_index()
output_df.columns = ['DATE', '평균가격']

print(f"전체 행 수: {len(output_df)}")        # 8766이어야 함
print(f"NaN 잔존: {output_df['평균가격'].isnull().sum()}개")  # 2~3개 (데이터 시작 전)

output_df.to_csv(PROCESSED_DIR + 'price_배추_가락시장_2002_2025.csv', encoding='utf-8-sig', index=False)

/Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/raw_data/2026년 신규수집 데이터/농넷 가락시장 경락가격/배추/
['농넷 가락시장 경락가격_일단위_10kg_상특중하_배추_2018.xlsx']
전체 행 수: 8766
NaN 잔존: 2개
